# PRE-stop: luật dừng unlearning dựa trên loss trước khi học forget set

Đích của luật là loss của mô hình ngay trước khi học F (mô hình chưa thấy F, cũng là loss mà retrain có trên F).
Quy trình: giai đoạn 1 (θ_pre) -> giai đoạn 2 (θ0, học thêm R2 và F) -> oracle θ_R2 x3 -> **Cổng 1 (chỉ cảnh báo)** -> quỹ đạo unlearning -> phân tích.

**Unlearning đếm theo EPOCH, đúng code gốc từng phương pháp** (`ga`, `scrub`, `salun`, `tarun`, `ft`): xem `PRE_STOP.md`.
Số epoch gốc: 10 (tarun: 2). Snapshot ở cuối mỗi epoch. Mọi số thời gian trong notebook là ước lượng chưa đo; ghi số đo thật vào RUN_NOTES khi chạy xong.

In [ ]:
import os, sys, json, glob, shutil, time

REPO_URL = "https://github.com/tiensinh2/Stopping_for_MU.git"
REPO_DIR = "/kaggle/working/ralm_repo"
OUT      = "/kaggle/working/pre_out"
PROFILE  = "kaggle"
SEEDS    = "0 1 2"      # 3 seed; các bước pre/cfg/traj tự bỏ qua phần đã có (dùng RESTORE_FROM để nạp seed 0 đã chạy). Hết giờ session thì chạy từng seed: "1", rồi "2"
FTYPES   = "random-450 random-1350 class-0 hard-450"
METHODS  = "ga scrub salun tarun ft"   # tarun chỉ chạy class-0, ft chỉ chạy random-450 (code gốc)
RESTORE_FROM = ""           # vd "/kaggle/input/<ten-notebook>/pre_out"

In [ ]:
%cd /kaggle/working
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", "https://%s@" % UserSecretsClient().get_secret("GH_TOKEN"))
except Exception:
    pass
if os.path.isdir(REPO_DIR + "/.git"):
    !cd {REPO_DIR} && git pull {url} main
else:
    !git clone {url} {REPO_DIR}
    !cd {REPO_DIR} && git remote set-url origin {REPO_URL}
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -r requirements.txt
if RESTORE_FROM and os.path.isdir(RESTORE_FROM):
    shutil.copytree(RESTORE_FROM, OUT, dirs_exist_ok=True)
    print("đã khôi phục", RESTORE_FROM, "->", OUT)
print("quỹ đạo đã có:", len(glob.glob(OUT + "/logs/traj_*_meta.json")))

In [ ]:
import torch
print("GPU:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Không thấy GPU: bật Accelerator = GPU rồi Restart session"
!python -m pytest tests/test_pre_stop.py -x -q

## 1. Giai đoạn 1: θ_pre trên R1 (100 epoch)
**GATE:** `pre` tự dừng nếu test acc của θ_pre < 90% (kịch bản mục 5). Khi đó gửi log, không đổi công thức.

In [ ]:
t0 = time.time()
!python -m pre_stop.run pre --profile {PROFILE} --out {OUT} --seeds {SEEDS}
print("xong sau %.0f phút" % ((time.time() - t0) / 60))

## 2. Giai đoạn 2 (θ0, cổng w_F), đóng băng P, oracle θ_R2 x3, rồi Cổng 1
Cổng 1 tính e_pre/nu trước khi unlearning. Nếu median e_pre/nu > 10 ở random-450 thì **chỉ in cảnh báo, vẫn chạy tiếp** (theo yêu cầu "cứ chạy ra kết quả rồi báo cáo sau"); kết quả luật phải đọc cùng bảng này.

In [ ]:
t0 = time.time()
!python -m pre_stop.run cfg   --profile {PROFILE} --out {OUT} --seeds {SEEDS} --ftypes {FTYPES}
!python -m pre_stop.run gate1 --profile {PROFILE} --out {OUT} --seeds {SEEDS} --ftypes {FTYPES}
print("xong sau %.0f phút" % ((time.time() - t0) / 60))

In [ ]:
import pandas as pd
g = pd.read_csv(OUT + "/results/gate1.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(g.round(3).to_string(index=False))

## 3. Quỹ đạo unlearning theo epoch
Không còn bước quét lr: optimizer, lr, batch, số epoch và chế độ BN lấy đúng code gốc của từng phương pháp (bảng trong `PRE_STOP.md`).
Mỗi quỹ đạo có epoch 0 (θ0) và một snapshot ở cuối mỗi epoch (10 epoch; tarun 2 epoch).

In [ ]:
t0 = time.time()
!python -m pre_stop.run traj --profile {PROFILE} --out {OUT} --seeds {SEEDS} --ftypes {FTYPES} --methods {METHODS}
print("xong sau %.0f phút" % ((time.time() - t0) / 60))
import glob as _g, json as _j
for m in sorted(_g.glob(OUT + "/logs/traj_*_meta.json")):
    d = _j.load(open(m, encoding="utf-8"))
    print(d["run_id"], "epoch", d["n_epochs"], "snapshot", d["n_snapshots"], "updates", d["n_updates"], "diverged", d["diverged"])

## 4. Phân tích
Median và IQR, mọi khoảng cách chuẩn hoá bằng nu (mục 9). Báo cáo gồm: bảng tiền đề, bảng chính theo luật, tiêu chí mục 12 cho đủ 8 biến thể PRE, điều kiện bác bỏ, kiểm định dấu.

In [ ]:
!python -m pre_stop.run analyze --profile {PROFILE} --out {OUT} --seeds {SEEDS} --ftypes {FTYPES} --methods {METHODS}
print(open(OUT + "/results/report_auto.md", encoding="utf-8").read())

## 5. Đóng gói
Tải `pre_out.zip` (không kèm checkpoint nặng) hoặc dùng làm `RESTORE_FROM` cho session sau.

In [ ]:
!cd /kaggle/working && zip -qr pre_out_light.zip pre_out -x "pre_out/ckpt/*" && ls -la pre_out_light.zip